# Entrenamiento de modelo early game

Este notebook carga `dataset_team_metrics_early_<EARLY_GAME_MINUTE>min.csv` generado desde timelines de MATCH-V5 y entrena un clasificador usando solo features de los primeros 15 minutos.

No se usan variables de fin de partida ni métricas posteriores al minuto 15.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
sns.set(style='whitegrid')


In [ ]:
from PipeLine_API_Riot.UsuarioRandom.configRandom import SETTINGS

minute = int(SETTINGS.get('EARLY_GAME_MINUTE', 10))
output_file = f'../PipeLine_API_Riot/dataset_team_metrics_early_{minute}min.csv'
df = pd.read_csv(output_file)
print('Loading early-game dataset for minute:', minute)
df.head()


In [ ]:
early_features = [
    'first_blood', 'early_dragons', 'early_heralds', 'early_item_purchases',
    'team_total_gold', 'team_avg_gold', 'team_total_xp', 'team_avg_xp',
    'team_total_minions', 'team_avg_minions', 'team_total_jungle_minions', 'team_avg_jungle_minions'
]
X = df[early_features].copy()
y = df['match_win'].astype(int)
X = X.fillna(X.median())
X.head()


In [ ]:
print('Distribución objetivo:')
print(y.value_counts(normalize=True))
print('Shape:', X.shape)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'xgboost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_estimators=100),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}
for name, model in models.items():
    acc = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
    auc = cross_val_score(model, X_scaled, y, cv=cv, scoring='roc_auc')
    print(f'{name}: accuracy={np.round(acc.mean(),4)} +- {np.round(acc.std(),4)}, auc={np.round(auc.mean(),4)} +- {np.round(auc.std(),4)}')


In [ ]:
best_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_estimators=200)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print('Accuracy test:', accuracy_score(y_test, y_pred))
print('ROC AUC test:', roc_auc_score(y_test, best_model.predict_proba(X_test)[:,1]))
print(classification_report(y_test, y_pred))
importances = best_model.feature_importances_
feat_imp = pd.Series(importances, index=early_features).sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='magma')
plt.title('Feature Importance - Early Game Metrics')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## Observaciones
- Este notebook usa únicamente métricas de los primeros 15 minutos.
- Si quieres comparar XGBoost vs Random Forest, ajusta los hiperparámetros y repite la validación cruzada.
- Si el dataset contiene pocas partidas, considera aumentar `PLAYERS_LIMIT` y `MATCHES_PER_PLAYER` en `configRandom.py`.